In [1]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

/mnt/c/Users/wstre/projects/lca-lc-foundations/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Write to state

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.8,
)

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [7]:
from pprint import pprint

pprint(response)
print(response["messages"][-1].content)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='ef9fff94-793b-4b61-a917-2fa14b7c037e'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user just said their favorite color is green. Let me check the tools available. There\'s a function called update_favourite_colour that takes a favorite_colour parameter. Since they\'ve revealed their color, I should call that function with "green" as the argument. I need to make sure the parameter is a string. Yep, that\'s all. No other tools are needed here. Just update the favorite color.\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Okay, the user just said their favorite color is green. Let me check the tools available. There\'s a function called update_favourite_colour that takes a favorite_colour parameter. Since they\'ve revealed their color, I should call that f

In [ ]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you? what is my favorite color?")],
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)
print(response["messages"][-1].content)

{'favourite_colour': 'red',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='ede69d2b-62c7-4e3b-b133-c945f9469843'),
              AIMessage(content="Hello! I'm doing well, thank you. Could you please share your favorite color with me so I can update it in your state? 😊", additional_kwargs={'reasoning_content': 'Okay, the user said, "Hello, how are you?" Let me think about how to respond.\n\nFirst, I need to check if there\'s a tool available that can help with this. The provided tool is update_favourite_colour, which requires the user to reveal their favorite color. But the user\'s message doesn\'t mention anything about their favorite color. They just greeted me and asked how I am.\n\nSince the tool needs the favorite_color parameter, and the user hasn\'t provided that information yet, I can\'t call the function. My response should be friendly and ask them about their favorite color to proceed. I shouldn\'t assume or make up a 

## Read state

In [10]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)
print(response["messages"][-1].content)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='288598c8-c0aa-4706-b57f-131c8d40cd38'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user just told me their favorite color is green. Let me check the tools available. There\'s a function called update_favourite_colour that takes a favourite_colour parameter. Since the user provided the color, I should call that function to update their preference. The read_favourite_colour function isn\'t needed here because they already stated their color. So I\'ll use update_favourite_colour with "green" as the argument.\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Okay, the user just told me their favorite color is green. Let me check the tools available. There\'s a function called update_favourite_colour that takes a favourite_colour parameter. Since the user provi

In [12]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)
print(response["messages"][-1].content)

KeyboardInterrupt: 